# Label Leakage Audit


This audit requires the raw `L1_events_v3.parquet` event file, which is excluded from the public release per the StatsBomb academic licence ([`../data/DATA_AVAILABILITY.md`](../data/DATA_AVAILABILITY.md)). Holders of an academic licence can regenerate the input via `../scripts/cache/prepare_l1_events.py`. The pre-computed result table in Section 3 below is also reported as Table S1 in `manuscript/supplementary.tex`.


## 1. Motivation

U1's val AUC reaches 0.99 for Ball Receipt and Dribble within the first two epochs. Two competing explanations are possible:

1. *Legitimate*: the 22 tabular features and the 7-channel SoccerMap CNN combine to give a near-perfect predictor (easy task with strong signal).
2. *Leakage*: one or more features encode the label directly (e.g. a column that reflects the recorded outcome rather than the pre-action context).

We discriminate between (1) and (2) by computing the max-direction single-feature AUC for every feature × task combination on the train split. If any feature alone reaches AUC > 0.90, the task is solvable from one column and leakage is suspected; otherwise the high task AUC must come from feature combinations and the CNN. The scan covers only the 22 tabular features, since the SoccerMap freeze-frame is, by construction, captured at event start and cannot encode the recorded outcome.


## 2. Method

For each binary task `t` and each feature `f`:
1. Restrict to events of `type_name == cfg.type_name` (e.g. all Pass events for the Pass task) and to TRAIN_SEASONS (22/23, 23/24; no val/test contamination).
2. Compute the label `y` exactly as in `scripts/training/train_unified.py`:
   - Pass: `pass_outcome_name.isna()` (NaN = success)
   - Dribble: `dribble_outcome_name == 'Complete'`
   - Ball Receipt: `ball_receipt_outcome_name.isna()`
   - Shot (excl. penalties): `shot_outcome_name == 'Goal'`
   - Duel: `duel_outcome_name ∈ {'Won', 'Success In Play'}`
   - Interception: same as Duel
   - Ball Recovery: `ball_recovery_failure != True`
   - Pressure: turnover within next 2 on-ball actions (see `compute_pressure_labels`)
3. Compute `max-direction AUC = max(AUC(y, f), AUC(y, -f))`; covers the case where a feature has reversed-polarity predictive signal.
4. Flag any task whose top-1 feature reaches AUC > 0.90.

## 3. Result: single-feature AUC summary

Pre-computed from `scripts/audit/audit_label_leakage.py` (`audit/label_leakage_report.md`):

| Task | n_train | pos_rate | top feature | top AUC | 1-feat LR AUC | Flag |
|---|---:|---:|---|---:|---:|---|
| pass | 1,477,879 | 0.8055 | `pass_end_dist_to_goal` | 0.7175 | 0.7175 | OK |
| dribble | 42,383 | 0.5257 | `sb_distance_to_nearest_defender` | 0.5775 | 0.5775 | OK |
| ball_receipt | 1,407,718 | 0.8453 | `sb_distance_to_nearest_defender` | 0.7885 | 0.7885 | OK |
| shot | 38,516 | 0.0987 | `dist_to_goal` | 0.7156 | 0.7156 | OK |
| duel | 103,191 | 0.2881 | `sb_visible_opponents` | 0.6326 | 0.6326 | OK |
| interception | 29,740 | 0.5950 | `dist_to_goal` | 0.5563 | 0.5563 | OK |
| ball_recovery | 142,658 | 0.9230 | `sb_distance_to_nearest_defender` | 0.6906 | 0.6906 | OK |
| pressure | 484,267 | 0.0595 | `sb_distance_to_nearest_defender` | 0.5737 | 0.5737 | OK |

No task is flagged. The strongest single feature is `sb_distance_to_nearest_defender` for Ball Receipt at AUC = 0.7885; well below the leakage threshold of 0.90.

## 4. Reproduce the scan

The code below recomputes single-feature AUC for the Pass task as a smoke test. Full re-run is via `scripts/audit/audit_label_leakage.py` (~3 min).

In [2]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

PROJECT_ROOT = Path('..').resolve()
CACHE = PROJECT_ROOT / 'data'
PARQUET = CACHE / 'L1_events_v3.parquet'

if not PARQUET.exists():
    print(f'Skipped: {PARQUET.name} is excluded from the public release per the StatsBomb academic licence.')
    print(f'Pre-computed results for all 8 tasks are in the table above (Section 3).')
else:
    TRAIN_SEASONS = ['2_235_2022_23', '2_281_2023_24',
                     '11_235_2022_23', '11_281_2023_24']
    df = pd.read_parquet(PARQUET)
    df = df[df['season_dir'].isin(TRAIN_SEASONS)]
    df_pass = df[df['type_name'] == 'Pass']
    y = df_pass['pass_outcome_name'].isna().astype(int).values
    print(f'pass: n_train={len(y):,}  pos_rate={y.mean():.4f}')
    for f in ['pass_end_dist_to_goal', 'pass_is_progressive',
              'pass_length', 'sb_num_defenders_on_goal_side']:
        if f not in df_pass.columns:
            continue
        s = pd.to_numeric(df_pass[f], errors='coerce').values.astype(float)
        m = ~np.isnan(s)
        if m.sum() > 100:
            auc = max(roc_auc_score(y[m], s[m]), roc_auc_score(y[m], -s[m]))
            print(f'  {f:36s}  AUC={auc:.4f}')


Skipped: L1_events_v3.parquet is excluded from the public release per the StatsBomb academic licence.
Pre-computed results for all 8 tasks are in the table above (Section 3).


## 5. Interpretation - why Ball Receipt and Dribble reach 0.99 val AUC

Given that no single feature exceeds AUC 0.79, the val 0.99 comes from feature combinations and the SoccerMap CNN. Two regimes explain it:

*Ball Receipt* (pos_rate = 0.845, balanced 84/16):
- The strongest single feature (`sb_distance_to_nearest_defender`) already explains 0.79 AUC.
- Combining the four SB-360 features (defender distance, defenders on goal side, visible teammates/opponents) with the 7-channel freeze-frame yields a strong joint predictor.
- The 85% positive class is a soft easy-task bias: a predictor only needs to identify the rare failure cases (player surrounded by defenders) to reach high AUC.

*Dribble* (pos_rate = 0.524, balanced 52/48):
- The strongest single feature reaches only 0.58, yet the model reaches 0.99; the CNN does most of the lifting, consistent with Dribble being the task most reliant on spatial context.
- The 0.99 figure is the in-season val AUC. The held-out 2024/25 test AUC is 0.74; the 25 pp gap is the covariate-shift signal also documented in the val-vs-test diagnostic in `reproduce.ipynb`.


## 6. Conclusion

No label leakage detected on any of the eight binary tasks. All val AUC values are reachable through legitimate feature combinations and the SoccerMap CNN. The full-precision static report is `../scripts/audit/audit_label_leakage.py` output, summarised as Table S1 in the supplementary material.
